# Module 2: Training Speedups + Stability You Can Trust

Your training loop from Module 1 works, but it's leaving performance on the table.

In this module, you'll:

1. **Speed up training** with mixed precision (AMP), DataLoader tuning, and `torch.compile`
2. **Make it stable** with NaN detection, gradient monitoring, and OOM prevention
3. **Profile** to find real bottlenecks (not guessed ones)

---

In [ ]:
# --- Colab / Environment Setup (run this cell first) ---
import os, subprocess

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.exists("/content/pytorch-production-workshop"):
        subprocess.run(["git", "clone", "https://github.com/arj7192/pytorch-production-workshop.git"], cwd="/content", check=True)
    os.chdir("/content/pytorch-production-workshop/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete - GPU:", os.environ.get("COLAB_GPU", "not detected"))

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import time
import torch
import torch.nn as nn

from src.model import build_model
from src.data import prepare_wikitext2, create_dataloaders
from src.utils import set_seed, get_device, NaNDetector
from src.evaluate import evaluate

set_seed(42)
device = get_device()
print(f"Device: {device}")

In [ ]:
# Load data and model from Module 1
train_dataset, val_dataset, _, tokenizer = prepare_wikitext2(
    vocab_size=8192, seq_len=128, tokenizer_path='../tokenizer.json'
)

config = {
    'vocab_size': tokenizer.get_vocab_size(),
    'd_model': 256, 'n_heads': 4, 'd_ff': 512,
    'n_layers': 4, 'max_seq_len': 128, 'dropout': 0.1,
}
model = build_model(config).to(device)
print(f"Model: {model.count_parameters():,} parameters")

## 2.1 DataLoader Performance

Your GPU is expensive hardware. If it's sitting idle waiting for the CPU to prepare data, you're wasting money. This is the most common bottleneck in training pipelines - and the easiest to fix.

**Three knobs to tune:**

- **`num_workers`**: With 0 (default), the main process loads data synchronously - GPU waits after every batch. With 2+, separate processes prefetch data in parallel while the GPU computes. Think of it as a factory assembly line. **Caveat**: on macOS, `num_workers > 0` can deadlock due to fork-safety issues. On Colab, you usually get only 2 CPU cores, so `num_workers=2` is the max.

- **`pin_memory`**: When you call `.to('cuda')`, PyTorch copies data from pageable RAM to page-locked RAM, then DMA-transfers to GPU. `pin_memory=True` skips the first copy by allocating directly in page-locked memory. Pair with `non_blocking=True` on `.to()` for async transfer. Typically 10-30% faster data loading.

- **`persistent_workers`**: Without this, worker processes are killed and respawned every epoch. Each spawn costs hundreds of milliseconds. `persistent_workers=True` keeps them alive. Always use it when `num_workers > 0`.

**Important**: Always benchmark on YOUR hardware. Internet advice like "use `num_workers=4`" can make things slower on a 2-core machine.

In [ ]:
def benchmark_dataloader(dataset, batch_size, num_workers, pin_memory, n_batches=100):
    """Time how long it takes to iterate through n_batches."""
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
    )
    
    # Warmup
    it = iter(loader)
    for _ in range(min(5, n_batches)):
        next(it)
    
    # Benchmark
    start = time.perf_counter()
    it = iter(loader)
    for i in range(n_batches):
        batch = next(it)
        if pin_memory and device.type == 'cuda':
            batch[0].to(device, non_blocking=True)
            batch[1].to(device, non_blocking=True)
    elapsed = time.perf_counter() - start
    
    return elapsed / n_batches * 1000  # ms per batch


configs_to_test = [
    {'num_workers': 0, 'pin_memory': False},
    {'num_workers': 2, 'pin_memory': False},
    {'num_workers': 2, 'pin_memory': True},
    {'num_workers': 4, 'pin_memory': True},
]

print(f"{'Config':<40} {'ms/batch':>10}")
print('-' * 52)
for cfg in configs_to_test:
    try:
        ms = benchmark_dataloader(train_dataset, batch_size=64, **cfg)
        print(f"workers={cfg['num_workers']}, pin_memory={cfg['pin_memory']:<5}  {ms:>10.2f}")
    except Exception as e:
        print(f"workers={cfg['num_workers']}, pin_memory={cfg['pin_memory']:<5}  FAILED: {e}")

## 2.2 Automatic Mixed Precision (AMP)

This is probably the single highest-impact optimization for any GPU training script, and it's just a few lines of code.

**The idea**: NVIDIA GPUs since V100 (2017) have Tensor Cores that do float16 matrix math at 2-8x the speed of float32. But naively running everything in float16 causes training to explode - float16 can only represent numbers up to 65504, and tiny gradients (1e-8) round to zero.

**PyTorch's `autocast` is the solution**: It selectively runs operations in float16 or float32:
- **float16** (fast): matrix multiplications, convolutions, linear layers - the expensive stuff
- **float32** (safe): softmax, layer norm, loss computation, reductions - numerically sensitive ops

This automatic selection is what makes AMP safe. You get the speed of float16 without the instability.

**GradScaler**: Float16 gradients can underflow to zero deep in the network. The scaler multiplies the loss by a large factor (~65536) before backward, scaling all gradients up into float16's representable range. After backward, it divides them back down. If any gradient overflows, that step is skipped and the scale is reduced. Fully automatic.

> **Note on bfloat16**: A100+ GPUs support bfloat16, which has float32's exponent range and doesn't need GradScaler. The T4 on Colab doesn't support bfloat16, so we use float16 + GradScaler here.

In [ ]:
def train_epoch(model, loader, optimizer, device, use_amp=False, scaler=None, max_grad_norm=1.0):
    """Train one epoch with optional AMP."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for input_ids, targets in loader:
        input_ids = input_ids.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        # autocast selectively runs ops in float16 (matmul, linear) or float32
        # (softmax, layernorm, loss). enabled=False makes it a no-op on CPU.
        with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
            output = model(input_ids, targets=targets)
            loss = output['loss']
        
        if scaler is not None:
            # AMP path: scale loss -> backward -> unscale -> clip -> step -> update
            # ORDER MATTERS: unscale_ BEFORE clipping, otherwise you're clipping
            # against gradients that are 65536x their real magnitude.
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
        
        optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches


# --- Compare: FP32 vs AMP ---
train_loader, _ = create_dataloaders(train_dataset, val_dataset, batch_size=64)

results = {}
for mode, use_amp in [('FP32', False), ('AMP (FP16)', True)]:
    set_seed(42)
    model_bench = build_model(config).to(device)
    opt = torch.optim.AdamW(model_bench.parameters(), lr=3e-4)
    scaler = torch.amp.GradScaler('cuda') if (use_amp and device.type == 'cuda') else None
    
    # Warmup
    train_epoch(model_bench, train_loader, opt, device, use_amp=use_amp, scaler=scaler)
    
    start = time.perf_counter()
    loss = train_epoch(model_bench, train_loader, opt, device, use_amp=use_amp, scaler=scaler)
    elapsed = time.perf_counter() - start
    
    results[mode] = {'time': elapsed, 'loss': loss}
    print(f"{mode:15s} | time: {elapsed:.1f}s | loss: {loss:.4f}")

if len(results) == 2:
    speedup = results['FP32']['time'] / results['AMP (FP16)']['time']
    print(f"\nAMP speedup: {speedup:.2f}x")

## 2.3 `torch.compile` (PyTorch 2.x)

One line of code: `model = torch.compile(model)`. Behind the scenes, PyTorch traces the computation graph, identifies sequences of operations that can be fused into single GPU kernels (e.g., LayerNorm + Dropout + Add becomes one kernel instead of three), and generates optimized CUDA code via the Triton compiler. The memory bandwidth savings from reduced kernel launches are substantial.

**Gotchas:**
- **First forward pass is slow** (30-60s) - that's the compilation. Every subsequent pass uses the compiled kernels. This is a one-time cost that's amortized over the full training run.
- **Dynamic shapes trigger recompilation** - if your sequence length changes every batch, the compiler recompiles for each new shape. Use fixed or bucketed sizes. Our model uses fixed seq_len=128, which is ideal.
- **Not all ops are supported** - torch.compile may fail on some platforms (macOS, older GPUs). The cell catches this gracefully.

> **"Not enough SMs" warning**: The T4 has 40 Streaming Multiprocessors. The compiler's most aggressive tuning mode wants more. It falls back to a slightly less optimized strategy. The compiled code is still faster than eager.

In [ ]:
set_seed(42)
model_compiled = build_model(config).to(device)

# torch.compile - the one-line speedup
try:
    model_compiled = torch.compile(model_compiled)
    print("Model compiled successfully")
    
    opt = torch.optim.AdamW(model_compiled.parameters(), lr=3e-4)
    
    # First epoch is slow (compilation)
    print("Compiling (first epoch)...")
    start = time.perf_counter()
    train_epoch(model_compiled, train_loader, opt, device)
    compile_time = time.perf_counter() - start
    print(f"  First epoch (includes compilation): {compile_time:.1f}s")
    
    # Subsequent epochs are fast
    start = time.perf_counter()
    loss = train_epoch(model_compiled, train_loader, opt, device)
    fast_time = time.perf_counter() - start
    print(f"  Second epoch (compiled):            {fast_time:.1f}s | loss: {loss:.4f}")

except Exception as e:
    print(f"torch.compile not available or failed: {e}")
    print("This is fine - torch.compile requires PyTorch 2.x and may not work on all platforms.")

## 2.4 Training Stability: Detecting and Fixing Failures

Production training runs often fail silently. You come back after 8 hours to find the loss went NaN at hour 2 and you've been wasting GPU time ever since. These three failure modes account for most overnight training crashes:

1. **NaN loss** - Usually from learning rate too high, bad data, or numerical overflow in float16 without GradScaler. Once a weight is NaN, it's viral - every computation involving it produces NaN. The model is irrecoverable; you must restore from a checkpoint.

2. **Exploding gradients** - Gradient norms spike (50 -> 500 -> 5000 -> NaN) over just a few steps. This is the precursor to NaN. Gradient clipping prevents it, but you should also monitor the pre-clip norm to detect when something is going wrong.

3. **OOM errors** - Batch too large for GPU memory. Unlike the other two, this crashes immediately rather than silently. But it's preventable with back-of-envelope math.

Let's demonstrate each failure mode and its fix. **These demos are intentionally destructive** - we'll watch a model die in real time, then show how simple safeguards prevent it.

In [ ]:
# --- Demo: Exploding gradients from high learning rate ---
# We use SGD (not Adam) to make the failure obvious. Adam's adaptive LR partially
# masks the problem by dividing gradients by their running variance. SGD shows
# the raw gradient dynamics. Watch the Max Grad column - it escalates fast.
set_seed(42)
model_unstable = build_model(config).to(device)
opt_bad = torch.optim.SGD(model_unstable.parameters(), lr=10.0)  # Way too high!

detector = NaNDetector()

print("Training with lr=10.0 (intentionally unstable)...")
print(f"{'Step':<8} {'Loss':>12} {'Max Grad':>12} {'Status':>10}")
print('-' * 46)

model_unstable.train()
for step, (x, y) in enumerate(train_loader):
    if step >= 20:
        break
    
    x, y = x.to(device), y.to(device)
    output = model_unstable(x, targets=y)
    loss = output['loss']
    
    loss.backward()
    
    grad_stats = detector.check_gradients(model_unstable)
    is_nan = detector.check_loss(loss, step)
    status = 'NaN!' if is_nan else ('WARN' if grad_stats['max_grad'] > 10 else 'OK')
    
    print(f"{step:<8} {loss.item():>12.4f} {grad_stats['max_grad']:>12.2f} {status:>10}")
    
    if is_nan:
        print("\n>>> Loss went NaN - this is what happens without gradient clipping!")
        break
    
    opt_bad.step()
    opt_bad.zero_grad()

In [ ]:
# --- Fix: Same setup but with gradient clipping ---
# clip_grad_norm_ computes the total L2 norm of all gradients across all
# parameters. If it exceeds max_norm, all gradients are scaled down proportionally
# (preserving direction, limiting step size). This is different from clip_grad_value_
# which clips each gradient independently and can distort the gradient direction.
set_seed(42)
model_stable = build_model(config).to(device)
opt_clipped = torch.optim.SGD(model_stable.parameters(), lr=1.0)  # Still high, but clipped

print("Training with lr=1.0 + gradient clipping (max_norm=1.0)...")
print(f"{'Step':<8} {'Loss':>12} {'Grad Norm':>12} {'Clipped?':>10}")
print('-' * 46)

model_stable.train()
for step, (x, y) in enumerate(train_loader):
    if step >= 20:
        break
    
    x, y = x.to(device), y.to(device)
    output = model_stable(x, targets=y)
    loss = output['loss']
    loss.backward()
    
    # Clip and capture the original norm
    grad_norm = torch.nn.utils.clip_grad_norm_(model_stable.parameters(), max_norm=1.0)
    clipped = 'YES' if grad_norm > 1.0 else 'no'
    
    print(f"{step:<8} {loss.item():>12.4f} {grad_norm.item():>12.2f} {clipped:>10}")
    
    opt_clipped.step()
    opt_clipped.zero_grad()

print("\n>>> Training stayed stable thanks to gradient clipping!")

In [ ]:
# --- Demo: OOM Prevention ---
# Do this math BEFORE training. An OOM crash at step 5000 after 2 hours
# of training is preventable with 30 seconds of arithmetic.
#
# The four memory consumers during training:
#   1. Parameters: 4 bytes per float32 value
#   2. Gradients: same size as parameters
#   3. Optimizer states: Adam keeps 2 buffers per param (momentum + variance) = 2x params
#   4. Activations: scales with batch_size * seq_len * layers (the big variable)
#
# For a 7B param model: params alone = 28 GB. Already over a 24 GB GPU!
# That's why large model training needs AMP, gradient checkpointing, or model parallelism.

def estimate_memory(model, batch_size, seq_len, dtype=torch.float32):
    """Rough estimate of peak training memory."""
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    grad_bytes = param_bytes
    optimizer_bytes = param_bytes * 2  # Adam: momentum + variance
    activation_bytes = batch_size * seq_len * model.d_model * 4 * model.transformer.num_layers
    
    total_mb = (param_bytes + grad_bytes + optimizer_bytes + activation_bytes) / 1024 / 1024
    return {
        'params_mb': param_bytes / 1024 / 1024,
        'grads_mb': grad_bytes / 1024 / 1024,
        'optimizer_mb': optimizer_bytes / 1024 / 1024,
        'activations_mb': activation_bytes / 1024 / 1024,
        'total_mb': total_mb,
    }

for bs in [32, 64, 128, 256, 512]:
    mem = estimate_memory(model, bs, 128)
    print(f"batch_size={bs:>4d}  →  ~{mem['total_mb']:>8.1f} MB  "
          f"(params: {mem['params_mb']:.1f}, activations: {mem['activations_mb']:.1f})")

## 2.5 PyTorch Profiler

Everyone has intuitions about where the bottleneck is. "It's the attention." "It's data loading." 80% of those intuitions are wrong. **Don't guess - measure.**

The PyTorch Profiler instruments every CUDA kernel launch, every CPU operation, and every memory allocation. The output table shows you exactly what's slow.

**How to read the table:**
- `aten::copy_` dominating? Your bottleneck is CPU-to-GPU data transfer, not the model. Fix: `pin_memory`, `non_blocking`, more `num_workers`.
- `aten::mm` dominating? Your model is compute-bound. Good. AMP and `torch.compile` help here.
- `cudaStreamSynchronize` large? You have unnecessary sync points. Common cause: calling `.item()` on a loss tensor every step (forces CPU-GPU sync). Log every N steps instead.

The Chrome trace (`chrome://tracing`) shows a timeline with CPU on top and GPU on bottom. **Gaps in the GPU row are wasted compute** - the GPU is starving for work.

In [ ]:
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler

train_loader_profile, _ = create_dataloaders(train_dataset, val_dataset, batch_size=64)

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

activities = [ProfilerActivity.CPU]
if device.type == 'cuda':
    activities.append(ProfilerActivity.CUDA)

# The schedule controls when profiling is active:
#   wait=2:   skip first 2 steps (let things stabilize)
#   warmup=2: profile but discard (JIT warmup)
#   active=6: these 6 steps are actually measured
#   repeat=1: do this cycle once
with profile(
    activities=activities,
    schedule=schedule(wait=2, warmup=2, active=6, repeat=1),
    on_trace_ready=tensorboard_trace_handler('../logs/profiler'),
    record_shapes=True,    # Log tensor shapes for each op
    profile_memory=True,   # Track memory allocations
    with_stack=True,       # Capture Python call stacks
) as prof:
    for step, (x, y) in enumerate(train_loader_profile):
        if step >= 12:
            break
        x, y = x.to(device), y.to(device)
        output = model(x, targets=y)
        output['loss'].backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        prof.step()

print(prof.key_averages().table(sort_by='cpu_time_total', row_limit=15))

In [ ]:
# The tensorboard_trace_handler already saved a trace file.
# Let's find it and also export a standalone Chrome trace.
from pathlib import Path

trace_dir = Path('../logs/profiler')
existing = sorted(trace_dir.glob('*.json')) if trace_dir.exists() else []

try:
    prof.export_chrome_trace(str(trace_dir / 'trace.json'))
    print(f"Chrome trace saved to {trace_dir / 'trace.json'}")
except RuntimeError:
    if existing:
        print(f"Trace already saved by tensorboard_trace_handler: {existing[-1]}")
    else:
        print("Trace was already saved in the previous cell.")

print("\nTo view: open chrome://tracing in Chrome and load the .json file.")

## 2.6 Putting It All Together - Optimized Training Run

Now we combine everything: AMP (if GPU), gradient clipping, cosine schedule with warmup, and proper DataLoader settings. This is the run that produces the checkpoint Modules 3 and 4 will use.

Notice the code structure: `USE_AMP = device.type == 'cuda'` - no if/else branching in the training loop. The `autocast` context manager accepts `enabled=False` and becomes a no-op on CPU. The scaler is `None` on CPU so the `if scaler` branch handles it cleanly. Same code, any hardware.

**This is what a production training script looks like.** Not flashy. Every possible failure mode has a safety net. Every decision is measurable. Every artifact is saved.

> **Optimization order of operations** (for your own projects): (1) Get a correct loop first. (2) Add gradient clipping + NaN detection. (3) Add AMP. (4) Tune DataLoader workers. (5) Try `torch.compile`. (6) Profile if still unsatisfied. Never optimize before measuring. Never add complexity before confirming the simple version is correct.

In [ ]:
from src.utils import setup_logging, CheckpointManager, MetricsTracker
from src.evaluate import generate_sample

# --- Fresh model with production settings ---
set_seed(42)
model = build_model(config).to(device)

EPOCHS = 5
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200         # ~5% of total steps
MAX_GRAD_NORM = 1.0
USE_AMP = device.type == 'cuda'  # Auto-detect: AMP on GPU, disabled on CPU

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda') if USE_AMP else None  # None on CPU = skip scaler path
train_loader, val_loader = create_dataloaders(train_dataset, val_dataset, batch_size=64)

total_steps = len(train_loader) * EPOCHS

def cosine_with_warmup(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_with_warmup)

logger = setup_logging('../logs')
ckpt_manager = CheckpointManager('../checkpoints', keep_last_n=3)

print(f"Training for {EPOCHS} epochs ({total_steps} steps)")
print(f"AMP: {USE_AMP} | Device: {device}")
print(f"Checkpoint dir: ../checkpoints/")
print("-" * 50)

global_step = 0
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (input_ids, targets) in enumerate(train_loader):
        # non_blocking=True: CPU doesn't wait for the transfer to finish.
        # Combined with pin_memory, this overlaps data transfer with compute.
        input_ids = input_ids.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=USE_AMP, dtype=torch.float16):
            output = model(input_ids, targets=targets)
            loss = output['loss']

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        epoch_loss += loss.item()
        global_step += 1

    avg_loss = epoch_loss / (batch_idx + 1)
    val_metrics = evaluate(model, val_loader, device, use_amp=USE_AMP)
    epoch_time = time.time() - epoch_start

    if val_metrics['val_loss'] < best_val_loss:
        best_val_loss = val_metrics['val_loss']

    # Save model + optimizer state (needed for training resume) + config (for model reconstruction)
    ckpt_manager.save(model, optimizer, epoch + 1, global_step, val_metrics['val_loss'], config)
    # Qualitative check: loss numbers can lie, but generated text doesn't
    sample = generate_sample(model, tokenizer, 'The', device, max_new_tokens=30)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | {epoch_time:.1f}s | "
        f"train_loss {avg_loss:.4f} | val_loss {val_metrics['val_loss']:.4f} | "
        f"val_ppl {val_metrics['val_perplexity']:.1f}"
    )
    print(f"  Sample: {sample[:100]}")

print(f"\nDone. Best val_loss: {best_val_loss:.4f}")
print(f"Checkpoint saved: {ckpt_manager.latest()}")

## Key Takeaways

| Technique | When to use | Typical speedup |
|-----------|------------|----------------|
| `num_workers > 0` | GPU training with heavy data loading | 1.5-3x |
| `pin_memory=True` | CUDA | 1.1-1.3x |
| AMP (float16) | CUDA with Tensor Cores | 1.5-2x |
| `torch.compile` | PyTorch 2.x, stable shapes | 1.2-1.5x |
| Gradient clipping | Always | N/A (stability) |
| NaN detection | Always | N/A (reliability) |

**Production rule**: Enable AMP + gradient clipping + NaN detection by default. Tune `num_workers` and `batch_size` per hardware.

We now have a **trained checkpoint** saved at `../checkpoints/` - Module 3 will load it for inference optimization and export.

**Next up**: Module 3 - optimizing inference and exporting the model for deployment.